In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Mounted at /content/drive


In [ ]:
!pip install sacrebleu
!pip install --upgrade wandb
!pip install transformers==4.57.1
!pip install evaluate

In [ ]:
# !pip install -q -U bitsandbytes
# !pip install -q -U accelerate
# !pip install -q -U peft

In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

import sacrebleu
import math
import re
import os
# import wandb
# from kaggle_secrets import UserSecretsClient
from transformers import TrainerCallback
from pathlib import Path
import shutil

from datasets import load_dataset

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments#, BitsAndBytesConfig
# from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model, TaskType

from torch.cuda import temperature

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Fix the seed (for reproducibility).
def seed_everything(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed(seed)

seed_everything()

# Train Single Model

In [ ]:
class Config:
    OUTPUT_DIR = "/content/drive/MyDrive/Deep Past Challenge 26/models/aduah_LARGE_from_zero_v8"

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoConfig, AutoTokenizer

model = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-large")
tokenizer = AutoTokenizer.from_pretrained("google/mt5-large")
model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


MT5ForConditionalGeneration(
  (shared): Embedding(250112, 1024)
  (encoder): MT5Stack(
    (embed_tokens): Embedding(250112, 1024)
    (block): ModuleList(
      (0): MT5Block(
        (layer): ModuleList(
          (0): MT5LayerSelfAttention(
            (SelfAttention): MT5Attention(
              (q): Linear(in_features=1024, out_features=1024, bias=False)
              (k): Linear(in_features=1024, out_features=1024, bias=False)
              (v): Linear(in_features=1024, out_features=1024, bias=False)
              (o): Linear(in_features=1024, out_features=1024, bias=False)
              (relative_attention_bias): Embedding(32, 16)
            )
            (layer_norm): MT5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): MT5LayerFF(
            (DenseReluDense): MT5DenseGatedActDense(
              (wi_0): Linear(in_features=1024, out_features=2816, bias=False)
              (wi_1): Linear(in_features=1024, out_features=2816, bias=Fals

In [ ]:
model.generation_config.num_beams = 5
model.generation_config.length_penalty = 1.7


In [ ]:
dataset_directory = "/content/drive/MyDrive/Deep Past Challenge 26/data"
dataset = load_dataset("csv", data_files={"train": f"{dataset_directory}/train_dataset.csv", "validation": f"{dataset_directory}/val_set.csv"})

In [ ]:
MAX_SOURCE_LEN = 600
MAX_TARGET_LEN = 690

In [ ]:
def preprocess(batch):
    inputs = tokenizer(
        batch["transliteration"],
        max_length=MAX_SOURCE_LEN,
        padding="max_length",
        truncation=True,
    )

    # ByT5 does not need as_target_tokenizer()
    labels = tokenizer(
        batch["translation"],
        max_length=MAX_TARGET_LEN,
        padding="max_length",
        truncation=True,
    )

    # CRITICAL LINE
    labels["input_ids"] = [
        [(token if token != tokenizer.pad_token_id else -100) for token in seq]
        for seq in labels["input_ids"]
    ]

    inputs["labels"] = labels["input_ids"]
    return inputs

In [ ]:
dataset = dataset.map(preprocess, batched=True, remove_columns=dataset['train'].column_names)

Map:   0%|          | 0/14532 [00:00<?, ? examples/s]

Map:   0%|          | 0/198 [00:00<?, ? examples/s]

In [ ]:
def deep_past_score(predictions, references):
    """
    predictions: List[str]
    references:  List[str]
    """

    # Compute bleu score using list of predictions and list of reference
    bleu = sacrebleu.corpus_bleu(
        predictions,
        [references],
        tokenize="intl"
    ).score

    # Compute chrf score from predicitons and reference lists
    chrf = sacrebleu.corpus_chrf(
        predictions,
        [references],
        word_order=2  # chrF++
    ).score

    # geometric mean
    score = math.sqrt(bleu * chrf)
    return {
        "bleu": bleu,
        "chrf++": chrf,
        "score": score,
    }

In [ ]:
# Callback class to delete the inferoir checkpoint if two checkpoints exists at a point in time.
# This is to save disk space.
class KeepBestDeleteWorstCallback(TrainerCallback):

    def __init__(self, metric="eval_score"):
        self.metric = metric

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):

        output_dir = Path(args.output_dir)

        checkpoints = sorted(
            output_dir.glob("checkpoint-*"),
            key=lambda x: int(x.name.split("-")[-1])
        )

        # only act if two checkpoints already exist
        if len(checkpoints) != 4:
            return

        # gather evaluation scores from trainer history
        score_by_step = {}

        for entry in state.log_history:
            if self.metric in entry and "step" in entry:
                score_by_step[entry["step"]] = entry[self.metric]

        # find score for each checkpoint
        scored = []
        for ckpt in checkpoints:
            step = int(ckpt.name.split("-")[-1])
            score = score_by_step.get(step)

            if score is not None:
                scored.append((score, ckpt))

        # if scores missing, do nothing
        if len(scored) != 4:
            return

        # determine worst checkpoint
        scored.sort(key=lambda x: x[0])  # ascending
        worst_ckpt = scored[0][1]

        shutil.rmtree(worst_ckpt)


def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # Replace -100 so they can be decoded
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # Ensure predictions are within the valid token ID range
    # Values outside this range can cause OverflowError during decoding.
    # Convert to integer type and clip to [0, tokenizer.vocab_size - 1]
    # before decoding.
    preds = np.array(preds).astype(int) # Ensure integer type
    preds = np.clip(preds, 0, tokenizer.vocab_size - 1)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Strip whitespace (important)
    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    return deep_past_score(decoded_preds, decoded_labels)

class SaveEvery5EpochsFrom20Callback(TrainerCallback):

    def __init__(self, tokenizer, save_dir: str):
        self.tokenizer = tokenizer
        self.save_dir = Path(save_dir)

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        epoch = int(state.epoch)

        if epoch % 5 == 0:
            save_path = self.save_dir / f"checkpoint-epoch-{epoch}"
            save_path.mkdir(parents=True, exist_ok=True)

            model.save_pretrained(save_path)
            self.tokenizer.save_pretrained(save_path)

            print(f"[SaveCallback] Эпоха {epoch} → {save_path}")

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=Config.OUTPUT_DIR,
    predict_with_generate=True,
    generation_max_length=690,
    # no_repeat_ngram_size=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    num_train_epochs=45,
    per_device_train_batch_size=5,
    per_device_eval_batch_size=5,
    gradient_accumulation_steps=6,
    learning_rate=1.2e-4,
    #weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_steps=800,
    # max_steps=2000,
    fp16=False,
    # logging_steps=100,
    # eval_steps=100,
    # save_steps=100,
    save_total_limit=4,
    metric_for_best_model="score",
    report_to="none",
    # load_best_model_at_end=True,
    remove_unused_columns=False,

    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    dataloader_prefetch_factor=1,

    torch_compile=True,
    torch_compile_backend="inductor",
)

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        KeepBestDeleteWorstCallback(),
    ]
)

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        KeepBestDeleteWorstCallback(),
        # SaveEvery5EpochsFrom20Callback(      # ← добавляем
        #     tokenizer=tokenizer,
        #     save_dir=Config.OUTPUT_DIR
        # )
    ]
)
trainer.train()

Epoch,Training Loss,Validation Loss,Bleu,Chrf++,Score
1,No log,0.635673,8.306664,26.560835,14.853684
2,2.031000,0.416645,27.271128,47.217567,35.884207
3,0.568500,0.345363,34.931469,54.820494,43.760261
4,0.406900,0.318974,40.002090,56.987530,47.745369
5,0.342000,0.304979,40.570100,58.762057,48.826044
6,0.302000,0.296737,41.155213,58.660235,49.134249
7,0.273000,0.296978,42.120067,58.695110,49.721645
8,0.247800,0.293116,43.350506,60.474831,51.201704
9,0.228300,0.297684,44.454499,61.130785,52.130015
10,0.209000,0.299243,44.799771,61.419054,52.455310


KeyboardInterrupt: 

In [ ]:
# --- Save Model ---
# Important: the model saved here will be loaded in the next notebook.
trainer.save_model(Config.OUTPUT_DIR)
tokenizer.save_pretrained(Config.OUTPUT_DIR)
print(f"Model saved to {Config.OUTPUT_DIR}")


# Average Checkpoints

In [ ]:
import gc
from transformers import DataCollatorForSeq2Seq

def average_checkpoints(checkpoint_paths: list, weights: list = None) -> AutoModelForSeq2SeqLM:
    n = len(checkpoint_paths)
    if weights is None:
        weights = [1.0 / n] * n

    avg_state_dict = {}

    for i, (ckpt_path, w) in enumerate(zip(checkpoint_paths, weights)):
        print(f"Загружаем [{i+1}/{n}] weight={w:.2f}: {ckpt_path}")
        model = AutoModelForSeq2SeqLM.from_pretrained(ckpt_path)
        state_dict = model.state_dict()

        for key in state_dict:
            if key not in avg_state_dict:
                avg_state_dict[key] = torch.zeros_like(state_dict[key]).float()
            avg_state_dict[key] += w * state_dict[key].float()

        del model
        gc.collect()
        torch.cuda.empty_cache()
        print(f"  VRAM свободно: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9:.1f}GB")

    # Загружаем усреднённые веса
    print("Загружаем усреднённую модель...")
    result_model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint_paths[0])
    result_model.load_state_dict(avg_state_dict)
    return result_model


# ---- Усредняем два чекпоинта ----
ckpt_1 = "/content/drive/MyDrive/Deep Past Challenge 26/models/aduah_LARGE_from_zero_v8/checkpoint-4850"
ckpt_2 = "/content/drive/MyDrive/Deep Past Challenge 26/models/aduah_LARGE_from_zero_v7_continue_3/checkpoint-970"

averaged_model = average_checkpoints(
    checkpoint_paths=[ckpt_1, ckpt_2],
    weights=[0.5, 0.5]
)
tokenizer = AutoTokenizer.from_pretrained(ckpt_1)

# ---- Оцениваем через trainer.evaluate() ----
# model.generation_config.num_beams = 5
# model.generation_config.length_penalty = 1.7

training_args_eval = Seq2SeqTrainingArguments(
    output_dir="/tmp/eval_avg",
    predict_with_generate=True,
    generation_max_length=690,
    per_device_eval_batch_size=5,
    fp16=False,
    report_to="none",
    remove_unused_columns=False,
)

trainer_eval = Seq2SeqTrainer(
    model=averaged_model,
    args=training_args_eval,
    eval_dataset=dataset["validation"],
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=averaged_model),
    compute_metrics=compute_metrics,
)
trainer_eval.args.generation_num_beams = 5

print("Оцениваем усреднённую модель...")
metrics_avg = trainer_eval.evaluate()
print(f"Averaged (0.5/0.5): {metrics_avg}")

# ---- Для сравнения — оцениваем каждый чекпоинт отдельно ----
metrics_single_all = []
for ckpt_path, label in [(ckpt_1, "Чекпоинт 1"), (ckpt_2, "Чекпоинт 2")]:
    model_single = AutoModelForSeq2SeqLM.from_pretrained(ckpt_path)

    trainer_single = Seq2SeqTrainer(
        model=model_single,
        args=training_args_eval,
        eval_dataset=dataset["validation"],
        data_collator=DataCollatorForSeq2Seq(tokenizer, model=model_single),
        compute_metrics=compute_metrics,
    )
    trainer_single.args.generation_num_beams = 5

    metrics_single = trainer_single.evaluate()
    metrics_single_all.append(metrics_single)
    print(f"{label}: {metrics_single}")

    del model_single
    gc.collect()
    torch.cuda.empty_cache()

# ---- Итоговое сравнение ----
print("\n=== СРАВНЕНИЕ ===")
print(f"Чекпоинт 1 : score = {metrics_single_all[0]['eval_score']:.4f}")
print(f"Чекпоинт 2 : score = {metrics_single_all[1]['eval_score']:.4f}")
print(f"Averaged   : score = {metrics_avg['eval_score']:.4f}")

Загружаем [1/2] weight=0.50: /content/drive/MyDrive/Deep Past Challenge 26/models/aduah_LARGE_from_zero_v8/checkpoint-4850
  VRAM свободно: 82.3GB
Загружаем [2/2] weight=0.50: /content/drive/MyDrive/Deep Past Challenge 26/models/aduah_LARGE_from_zero_v7_continue_3/checkpoint-970
  VRAM свободно: 82.3GB
Загружаем усреднённую модель...
Оцениваем усреднённую модель...


Averaged (0.5/0.5): {'eval_loss': 0.3056432604789734, 'eval_model_preparation_time': 0.0062, 'eval_bleu': 45.39070959271189, 'eval_chrf++': 61.69822213987071, 'eval_score': 52.9199970099914, 'eval_runtime': 114.4694, 'eval_samples_per_second': 1.73, 'eval_steps_per_second': 0.349}


Чекпоинт 1: {'eval_loss': 0.2992440164089203, 'eval_model_preparation_time': 0.0062, 'eval_bleu': 44.799770799631915, 'eval_chrf++': 61.41905440097537, 'eval_score': 52.4553101210337, 'eval_runtime': 111.7716, 'eval_samples_per_second': 1.771, 'eval_steps_per_second': 0.358}


Чекпоинт 2: {'eval_loss': 0.3260941207408905, 'eval_model_preparation_time': 0.006, 'eval_bleu': 44.42105908313452, 'eval_chrf++': 61.230084720625065, 'eval_score': 52.15271048603532, 'eval_runtime': 114.4613, 'eval_samples_per_second': 1.73, 'eval_steps_per_second': 0.349}

=== СРАВНЕНИЕ ===
Чекпоинт 1 : score = 52.4553
Чекпоинт 2 : score = 52.1527
Averaged   : score = 52.9200


In [ ]:
save_path = "LARGE_MIXES_v1"

averaged_model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

('LARGE_MIXES_v1/tokenizer_config.json',
 'LARGE_MIXES_v1/special_tokens_map.json',
 'LARGE_MIXES_v1/added_tokens.json')

# Ensemble MBR

In [ ]:
"""
Ensemble MBR Pipeline
=====================
Этапы:
  1. Определение моделей (ModelConfig)
  2. Генерация кандидатов от каждой модели последовательно
  3. Сбор кандидатов в DataFrame
  4. MBR decoding — выбор лучшего кандидата
  5. (Опционально) Оценка на валидации
"""

import gc
import math
from dataclasses import dataclass, field
from typing import List, Optional, Dict

import pandas as pd
import numpy as np
import sacrebleu
import torch
from tqdm import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer


# ============================================================
# Этап 0: Конфигурация моделей
# ============================================================

@dataclass
class ModelConfig:
    """Описание одной модели для ensemble"""
    name: str                          # Человекочитаемое имя
    model_path: str                    # Путь к весам
    tokenizer_path: Optional[str] = None  # Путь к токенайзеру (если отличается)

    # Generation параметры
    num_beams: int = 12
    num_return: int = 5
    max_input_length: int = 600
    max_output_length: int = 690
    length_penalty: float = 1.7

    # Batch
    batch_size: int = 5

    # Доп. параметры загрузки
    torch_dtype: Optional[torch.dtype] = None  # например torch.bfloat16
    local_files_only: bool = True

    def __post_init__(self):
        if self.tokenizer_path is None:
            self.tokenizer_path = self.model_path


# ============================================================
# Этап 1: Генерация кандидатов от одной модели
# ============================================================

def generate_candidates_single_model(
    config: ModelConfig,
    texts: List[str],
    device: str = "cuda",
) -> List[List[str]]:
    """
    Загружает модель, генерирует кандидатов, освобождает память.

    Returns:
        List[List[str]]: для каждого текста — список из num_return кандидатов
    """
    print(f"\n{'='*60}")
    print(f"[{config.name}] Loading model from {config.model_path}")
    print(f"  beams={config.num_beams}, return={config.num_return}, "
          f"lp={config.length_penalty}, batch={config.batch_size}")
    print(f"{'='*60}")

    # Загрузка
    load_kwargs = {"local_files_only": config.local_files_only}
    if config.torch_dtype is not None:
        load_kwargs["torch_dtype"] = config.torch_dtype

    model = AutoModelForSeq2SeqLM.from_pretrained(config.model_path, **load_kwargs)
    model.to(device)
    model.eval()

    tokenizer = AutoTokenizer.from_pretrained(
        config.tokenizer_path, local_files_only=config.local_files_only
    )

    # Генерация
    all_candidates = []
    n_batches = math.ceil(len(texts) / config.batch_size)

    for i in tqdm(range(0, len(texts), config.batch_size),
                  total=n_batches, desc=f"[{config.name}] Generating"):
        batch_texts = texts[i : i + config.batch_size]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            max_length=config.max_input_length,
            padding=True,
            truncation=True,
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                num_beams=config.num_beams,
                num_return_sequences=config.num_return,
                max_length=config.max_output_length,
                length_penalty=config.length_penalty,
                early_stopping=True,
            )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        # Разбить по примерам: [batch_size * num_return] → batch_size × num_return
        for j in range(len(batch_texts)):
            start = j * config.num_return
            end = start + config.num_return
            cands = [d.strip() for d in decoded[start:end]]
            all_candidates.append(cands)

    # Освобождение памяти
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    mem_free = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()
    print(f"[{config.name}] Done. Generated {len(all_candidates)} × {config.num_return} candidates. "
          f"VRAM free: {mem_free / 1e9:.1f} GB")

    return all_candidates


# ============================================================
# Этап 2: Прогон всех моделей, сбор кандидатов в DataFrame
# ============================================================

def generate_all_candidates(
    model_configs: List[ModelConfig],
    texts: List[str],
    device: str = "cuda",
) -> pd.DataFrame:
    """
    Последовательно прогоняет все модели.

    Returns:
        DataFrame с колонками:
          - text_idx: индекс текста
          - input_text: исходный текст
          - {model_name}_0, {model_name}_1, ...: кандидаты от каждой модели
    """
    n = len(texts)

    # Базовый DataFrame
    df = pd.DataFrame({
        "text_idx": range(n),
        "input_text": texts,
    })

    # Прогон каждой модели
    for config in model_configs:
        candidates = generate_candidates_single_model(config, texts, device)

        # Добавить колонки: model_name_0, model_name_1, ...
        for k in range(config.num_return):
            col_name = f"{config.name}_{k}"
            df[col_name] = [candidates[i][k] if k < len(candidates[i]) else ""
                           for i in range(n)]

    return df


def get_candidate_columns(df: pd.DataFrame) -> List[str]:
    """Вернуть имена колонок с кандидатами"""
    return [c for c in df.columns if c not in ("text_idx", "input_text")]


def get_candidates_for_row(row: pd.Series, candidate_cols: List[str]) -> List[str]:
    """Извлечь всех кандидатов для одного примера"""
    return [row[c] for c in candidate_cols if row[c] and str(row[c]).strip()]


# ============================================================
# Этап 3: MBR Decoding
# ============================================================

def mbr_decode_chrf(candidates: List[str]) -> str:
    """MBR по chrF++ (быстрее, стабильнее)"""
    if len(candidates) <= 1:
        return candidates[0] if candidates else ""

    # Дедупликация
    seen = set()
    unique = []
    for c in candidates:
        if c not in seen:
            seen.add(c)
            unique.append(c)
    candidates = unique

    if len(candidates) == 1:
        return candidates[0]

    best_score = -1
    best_text = candidates[0]

    for i, hyp in enumerate(candidates):
        total = 0
        for j, ref in enumerate(candidates):
            if i != j:
                total += sacrebleu.sentence_chrf(hyp, [ref], word_order=2).score
        avg = total / (len(candidates) - 1)

        if avg > best_score:
            best_score = avg
            best_text = hyp

    return best_text


def mbr_decode_combined(candidates: List[str]) -> str:
    """MBR по geometric mean(BLEU, chrF++) — как метрика соревнования"""
    if len(candidates) <= 1:
        return candidates[0] if candidates else ""

    seen = set()
    unique = []
    for c in candidates:
        if c not in seen:
            seen.add(c)
            unique.append(c)
    candidates = unique

    if len(candidates) == 1:
        return candidates[0]

    best_score = -1
    best_text = candidates[0]

    for i, hyp in enumerate(candidates):
        chrf_total = 0
        bleu_total = 0

        for j, ref in enumerate(candidates):
            if i != j:
                chrf_total += sacrebleu.sentence_chrf(hyp, [ref], word_order=2).score
                bleu_total += sacrebleu.sentence_bleu(hyp, [ref], tokenize="intl").score

        n = len(candidates) - 1
        avg_chrf = chrf_total / n
        avg_bleu = bleu_total / n

        if avg_bleu > 0 and avg_chrf > 0:
            combined = math.sqrt(avg_bleu * avg_chrf)
        else:
            combined = avg_chrf

        if combined > best_score:
            best_score = combined
            best_text = hyp

    return best_text



MBR_STRATEGIES = {
    "chrf": mbr_decode_chrf,
    "combined": mbr_decode_combined,
}


def select_best_candidates(
    candidates_df: pd.DataFrame,
    strategy: str = "chrf",
) -> pd.DataFrame:
    """
    Применяет MBR decoding к DataFrame с кандидатами.

    Returns:
        DataFrame с колонками: text_idx, input_text, prediction
    """
    decode_fn = MBR_STRATEGIES[strategy]
    candidate_cols = get_candidate_columns(candidates_df)

    print(f"\nMBR Decoding (strategy={strategy})")
    print(f"  Candidates per example: {len(candidate_cols)}")
    print(f"  Total examples: {len(candidates_df)}")

    predictions = []
    for _, row in tqdm(candidates_df.iterrows(),
                       total=len(candidates_df), desc="MBR Decoding"):
        cands = get_candidates_for_row(row, candidate_cols)
        best = decode_fn(cands)
        predictions.append(best)

    result = pd.DataFrame({
        "text_idx": candidates_df["text_idx"],
        "input_text": candidates_df["input_text"],
        "prediction": predictions,
    })

    return result


# ============================================================
# Этап 4: Оценка (валидация)
# ============================================================

def deep_past_score(predictions: List[str], references: List[str]) -> Dict[str, float]:
    """Метрика соревнования: geometric mean(BLEU, chrF++)"""
    bleu = sacrebleu.corpus_bleu(predictions, [references], tokenize="intl").score
    chrf = sacrebleu.corpus_chrf(predictions, [references], word_order=2).score
    score = math.sqrt(bleu * chrf) if bleu > 0 and chrf > 0 else 0.0
    return {"bleu": bleu, "chrf++": chrf, "score": score}


def evaluate_predictions(
    predictions_df: pd.DataFrame,
    references: List[str],
) -> Dict[str, float]:
    """Оценить предсказания"""
    preds = predictions_df["prediction"].tolist()
    metrics = deep_past_score(preds, references)

    print(f"\n{'='*40}")
    print(f"BLEU:   {metrics['bleu']:.4f}")
    print(f"chrF++: {metrics['chrf++']:.4f}")
    print(f"Score:  {metrics['score']:.4f}")
    print(f"{'='*40}")

    return metrics


def evaluate_per_model(
    candidates_df: pd.DataFrame,
    references: List[str],
    model_configs: List[ModelConfig],
) -> pd.DataFrame:
    """Оценить top-1 от каждой модели отдельно (для сравнения)"""
    results = []

    for config in model_configs:
        col = f"{config.name}_0"  # top-1 кандидат
        if col in candidates_df.columns:
            preds = candidates_df[col].tolist()
            metrics = deep_past_score(preds, references)
            metrics["model"] = config.name
            results.append(metrics)
            print(f"  {config.name}: score={metrics['score']:.4f} "
                  f"(BLEU={metrics['bleu']:.2f}, chrF++={metrics['chrf++']:.2f})")

    return pd.DataFrame(results)

In [ ]:
# ============================================================
# MBR Decoding
# ============================================================

def _dedup(candidates: List[str]) -> List[str]:
    """Дедупликация с сохранением порядка"""
    seen = set()
    unique = []
    for c in candidates:
        c = str(c).strip()
        if c and c not in seen:
            seen.add(c)
            unique.append(c)
    return unique


# chrF++ only

def mbr_decode_chrf(candidates: List[str]) -> str:
    """MBR по chrF++ (быстрее, стабильнее)"""
    if len(candidates) <= 1:
        return candidates[0] if candidates else ""

    # Дедупликация
    seen = set()
    unique = []
    for c in candidates:
        if c not in seen:
            seen.add(c)
            unique.append(c)
    candidates = unique

    if len(candidates) == 1:
        return candidates[0]

    best_score = -1
    best_text = candidates[0]

    for i, hyp in enumerate(candidates):
        total = 0
        for j, ref in enumerate(candidates):
            if i != j:
                total += sacrebleu.sentence_chrf(hyp, [ref], word_order=2).score
        avg = total / (len(candidates) - 1)

        if avg > best_score:
            best_score = avg
            best_text = hyp

    return best_text


def mbr_decode_combined(candidates: List[str]) -> str:
    """MBR по geometric mean(BLEU, chrF++) — как метрика соревнования"""
    if len(candidates) <= 1:
        return candidates[0] if candidates else ""

    seen = set()
    unique = []
    for c in candidates:
        if c not in seen:
            seen.add(c)
            unique.append(c)
    candidates = unique

    if len(candidates) == 1:
        return candidates[0]

    best_score = -1
    best_text = candidates[0]

    for i, hyp in enumerate(candidates):
        chrf_total = 0
        bleu_total = 0

        for j, ref in enumerate(candidates):
            if i != j:
                chrf_total += sacrebleu.sentence_chrf(hyp, [ref], word_order=2).score
                bleu_total += sacrebleu.sentence_bleu(hyp, [ref], tokenize="intl").score

        n = len(candidates) - 1
        avg_chrf = chrf_total / n
        avg_bleu = bleu_total / n

        if avg_bleu > 0 and avg_chrf > 0:
            combined = math.sqrt(avg_bleu * avg_chrf)
        else:
            combined = avg_chrf

        if combined > best_score:
            best_score = combined
            best_text = hyp

    return best_text



# Baseline-style weighted MBR ----

def _jaccard(a: str, b: str) -> float:
    """Token-level Jaccard similarity (0-100)"""
    ta = set(a.lower().split())
    tb = set(b.lower().split())
    if not ta and not tb:
        return 100.0
    if not ta or not tb:
        return 0.0
    return 100.0 * len(ta & tb) / len(ta | tb)


def _length_bonus(lengths: List[int], idx: int) -> float:
    """Гауссов бонус: близкие к медиане длины получают больше"""
    if not lengths:
        return 100.0
    median = float(np.median(lengths))
    sigma = max(median * 0.4, 5.0)
    z = (lengths[idx] - median) / sigma
    return 100.0 * math.exp(-0.5 * z * z)


def mbr_decode_baseline(
    candidates: List[str],
    w_chrf: float = 0.55,
    w_bleu: float = 0.25,
    w_jaccard: float = 0.20,
    w_length: float = 0.10,
    pool_cap: int = 32,
) -> str:
    """
    Baseline MBR: взвешенная комбинация chrF++ + BLEU + Jaccard + length bonus.
    """
    candidates = _dedup(candidates)
    if pool_cap:
        candidates = candidates[:pool_cap]

    n = len(candidates)
    if n == 0:
        return ""
    if n == 1:
        return candidates[0]

    pw_total = max(w_chrf + w_bleu + w_jaccard, 1e-9)
    lengths = [len(c.split()) for c in candidates]

    best_score = -1
    best_text = candidates[0]

    for i in range(n):
        pairwise_sum = 0
        for j in range(n):
            if j != i:
                chrf = sacrebleu.sentence_chrf(candidates[i], [candidates[j]], word_order=2).score
                bleu = sacrebleu.sentence_bleu(candidates[i], [candidates[j]], tokenize="intl").score
                jacc = _jaccard(candidates[i], candidates[j])

                pairwise_sum += (w_chrf * chrf + w_bleu * bleu + w_jaccard * jacc) / pw_total

        avg_pairwise = pairwise_sum / (n - 1)
        lb = _length_bonus(lengths, i)
        total = avg_pairwise + w_length * lb

        if total > best_score:
            best_score = total
            best_text = candidates[i]

    return best_text




MBR_STRATEGIES = {
    "chrf": mbr_decode_chrf,
    "combined": mbr_decode_combined,
    "baseline_chrf_heavy": lambda c: mbr_decode_baseline(c, w_chrf=0.80, w_bleu=0.10, w_jaccard=0.10, w_length=0.05),
}

In [ ]:
%%time

# --- Этап 0: Определить модели ---
model_configs = [
    ModelConfig(
        name="mix_v2",
        model_path="/content/drive/MyDrive/Deep Past Challenge 26/models/LARGE_MIXES/LARGE_MIXES_WITH_OLD_NEW_DATA_v2",
        num_beams=6,
        num_return=3,
        length_penalty=1.7,
    ),
    ModelConfig(
        name="drop_v1",
        model_path="/content/drive/MyDrive/Deep Past Challenge 26/models/LARGE_MIXES/LARGE_MIXES_TRAINED_WITH_TEST_DROP_v1",
        num_beams=6,
        num_return=3,
        length_penalty=1.7,
    ),
    ModelConfig(
        name="mt5_large_drop",
        model_path="/content/drive/MyDrive/Deep Past Challenge 26/models/aduah_MT_LARGE_from_zero_v1_continue_drop/checkpoint-1455",
        num_beams=6,
        num_return=3,
        length_penalty=1.7,
    ),
    ModelConfig(
        name="mt5_xl_bf16",
        model_path="/content/drive/MyDrive/Deep Past Challenge 26/models/XL_runpod_checkpoint-8190_bf16",
        num_beams=5,
        num_return=1,
        length_penalty=1.7,
        torch_dtype=torch.bfloat16
    ),
]

# --- Загрузить валидационные данные ---
raw_val = pd.read_csv("/content/drive/MyDrive/Deep Past Challenge 26/data/val_set.csv")
texts = raw_val["transliteration"].tolist()
references = raw_val["translation"].tolist()

device = "cuda" if torch.cuda.is_available() else "cpu"

# --- Этап 1-2: Генерация кандидатов ---
candidates_df = generate_all_candidates(model_configs, texts, device)

# Посмотреть что получилось
print(f"\nКандидаты DataFrame: {candidates_df.shape}")
print(f"Колонки: {candidates_df.columns.tolist()}")
# print(candidates_df.head(2))

# --- Этап 3: Оценка отдельных моделей (top-1) ---
print("\n--- Per-model scores (top-1) ---")
per_model = evaluate_per_model(candidates_df, references, model_configs)

# --- Этап 4: MBR Decoding ---
# Попробовать обе стратегии
for strategy in ["chrf", "combined", "baseline_chrf_heavy"]:
    predictions_df = select_best_candidates(candidates_df, strategy=strategy)
    print(f"\n--- MBR strategy: {strategy} ---")
    metrics = evaluate_predictions(predictions_df, references)

# --- Сохранить кандидатов для анализа ---
candidates_df.to_csv("candidates_val.csv", index=False)


[mix_v2] Loading model from /content/drive/MyDrive/Deep Past Challenge 26/models/LARGE_MIXES/LARGE_MIXES_WITH_OLD_NEW_DATA_v2
  beams=6, return=3, lp=1.7, batch=5


[mix_v2] Generating: 100%|██████████| 40/40 [01:48<00:00,  2.72s/it]


[mix_v2] Done. Generated 198 × 3 candidates. VRAM free: 102.0 GB

[drop_v1] Loading model from /content/drive/MyDrive/Deep Past Challenge 26/models/LARGE_MIXES/LARGE_MIXES_TRAINED_WITH_TEST_DROP_v1
  beams=6, return=3, lp=1.7, batch=5


[drop_v1] Generating: 100%|██████████| 40/40 [01:46<00:00,  2.66s/it]


[drop_v1] Done. Generated 198 × 3 candidates. VRAM free: 102.0 GB

[mt5_large_drop] Loading model from /content/drive/MyDrive/Deep Past Challenge 26/models/aduah_MT_LARGE_from_zero_v1_continue_drop/checkpoint-1455
  beams=6, return=3, lp=1.7, batch=5


...


--- MBR strategy: chrf ---

BLEU:   47.3968
chrF++: 63.3839
Score:  54.8105

MBR Decoding (strategy=combined)
  Candidates per example: 10
  Total examples: 198


MBR Decoding: 100%|██████████| 198/198 [00:04<00:00, 44.01it/s]



--- MBR strategy: combined ---

BLEU:   46.9952
chrF++: 63.0605
Score:  54.4384

MBR Decoding (strategy=baseline_chrf_heavy)
  Candidates per example: 10
  Total examples: 198


MBR Decoding: 100%|██████████| 198/198 [00:04<00:00, 42.02it/s]



--- MBR strategy: baseline_chrf_heavy ---

BLEU:   47.4057
chrF++: 63.3646
Score:  54.8073
CPU times: user 6min 27s, sys: 4.43 s, total: 6min 31s
Wall time: 6min 32s
